<a href="https://colab.research.google.com/github/chevasatrio/datamining_python_tugas2/blob/main/tugas_2_datamining_cheva_satrio_utomo_102062400031.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

# --- TAHAP 0: DATA INGESTION & DEFENSIVE CLEANING ---
# Membaca data dan mengantisipasi kecerobohan pengetikan di CSV
df = pd.read_csv('data_dm.csv')

# Membersihkan spasi tersembunyi di nama kolom (misal: ' HbAlc ' menjadi 'HbAlc')
df.columns = df.columns.str.strip()

# Memaksa standarisasi nama kolom agar sesuai dengan instruksi soal (HBA1c dan RGB).
# Mengakomodasi typo umum seperti HbAlc (L kecil), HbA1c, dan RBG (yang tertukar jadi RGB di soal).
rename_map = {
    'HbAlc': 'HBA1c',
    'HbA1c': 'HBA1c',
    'HBA1C': 'HBA1c',
    'RBG': 'RGB'
}
df.rename(columns=rename_map, inplace=True)

print("Kolom yang berhasil dideteksi dan distandarisasi:")
print(df.columns.tolist())
print("-" * 50)

# --- TAHAP 1-3: DATA CLEANING (GLOBAL & PER KELOMPOK) ---
# Memastikan kolom numerik tidak terbaca sebagai string karena typo
num_cols_to_fix = ['Hemoglobin', 'Hematokrit', 'Lekosit', 'Eritrosit', 'Trombosit', 'HBA1c', 'RGB']
for col in num_cols_to_fix:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Imputasi nilai kosong (NaN) dengan median spesifik per kategori diagnosis
for col in num_cols_to_fix:
    if col in df.columns:
        df[col] = df.groupby('Diagnose')[col].transform(lambda x: x.fillna(x.median()))

# Hapus baris jika variabel identitas utama masih kosong
df.dropna(subset=['Gender', 'Age', 'Diagnose'], inplace=True)


# --- TAHAP 4: DETEKSI & PENGHAPUSAN OUTLIER ---
def remove_outliers(df_in, col_name):
    if col_name not in df_in.columns:
        return df_in
    q1 = df_in[col_name].quantile(0.25)
    q3 = df_in[col_name].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return df_in[(df_in[col_name] >= lower_bound) & (df_in[col_name] <= upper_bound)]

for col in num_cols_to_fix:
    df = remove_outliers(df, col)


# --- TAHAP 5: PENANGANAN DUPLIKASI ---
df.drop_duplicates(inplace=True)


# --- TAHAP 6: PELABELAN (ENCODING) ---
# O untuk DM Type 2, 1 untuk DM Type 2 + Penyerta
df['Diagnose'] = df['Diagnose'].map({'DM TYPE 2': 0, 'DM TYPE 2 + Penyerta': 1})
# O untuk Laki-laki, 1 untuk Perempuan
df['Gender'] = df['Gender'].map({'L': 0, 'P': 1})

# Hapus data yang gagal dilabeli (typo parah pada kolom Diagnose/Gender)
df.dropna(subset=['Diagnose', 'Gender'], inplace=True)


# --- TAHAP 7: STANDARISASI MIN-MAX SCALER ---
if 'HBA1c' in df.columns and 'RGB' in df.columns:
    minmax = MinMaxScaler()
    df[['HBA1c', 'RGB']] = minmax.fit_transform(df[['HBA1c', 'RGB']])


# --- TAHAP 8: STANDARISASI Z-SCORE ---
cols_to_zscore = ['Hemoglobin', 'Hematokrit', 'Lekosit', 'Eritrosit', 'Trombosit']
# Pastikan kolom ada sebelum distandarisasi
cols_to_zscore_exist = [c for c in cols_to_zscore if c in df.columns]

if cols_to_zscore_exist:
    scaler_z = StandardScaler()
    df[cols_to_zscore_exist] = scaler_z.fit_transform(df[cols_to_zscore_exist])


# --- TAHAP 9: DATA REDUCTION (PCA) ---
if len(cols_to_zscore_exist) == 5:
    pca = PCA(n_components=2)
    blood_components = pca.fit_transform(df[cols_to_zscore_exist])
    df['Blood_PCA_1'] = blood_components[:, 0]
    df['Blood_PCA_2'] = blood_components[:, 1]


# --- TAHAP 10: FEATURE SELECTION ---
final_features = ['Gender', 'Age', 'Diagnose', 'HBA1c', 'RGB']
# Pastikan hanya memilih fitur yang benar-benar ada di dataframe
final_features_exist = [c for c in final_features if c in df.columns]

df_final = df[final_features_exist]

print("=== HASIL AKHIR DATA PREPARATION ===")
print(df_final.head(10))
print("\nDimensi data akhir:", df_final.shape)

Kolom yang berhasil dideteksi dan distandarisasi:
['Gender', 'Age', 'Diagnose', 'Hemoglobin', 'Hematokrit', 'Lekosit', 'Eritrosit', 'Trombosit', 'HBA1c', 'RGB']
--------------------------------------------------
=== HASIL AKHIR DATA PREPARATION ===
    Gender  Age  Diagnose     HBA1c       RGB
0        0  1.0         0  0.439560  0.183007
2        1  1.0         1  0.285714  0.197386
3        1  1.0         1  0.461538  0.542484
4        0  1.0         0  0.483516  0.309804
5        1  1.0         0  0.494505  0.406536
6        1  1.0         1  0.494505  0.507190
7        0  1.0         0  0.527473  0.756863
8        1  1.0         0  0.626374  0.396078
9        1  1.0         1  0.670330  0.814379
10       1  1.0         0  0.726374  0.379085

Dimensi data akhir: (1219, 5)
